# LoRA Training — Wan 2.1 I2V 480P
**วิธีใช้:**
1. Runtime → Change runtime type → T4 GPU
2. วาง clips ไว้ใน Google Drive ที่ `MyDrive/COMP_ANI/clips/`
3. รันทีละ cell ตามลำดับ

> ⚠️ Free Colab disconnect ทุก ~12 ชม. — ถ้า training ยังไม่เสร็จต้อง resume จาก checkpoint

In [ ]:
# CELL 1 — เช็ค GPU
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# CELL 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# เช็คว่าเจอ clips
import os
CLIPS_PATH = '/content/drive/MyDrive/COMP_ANI/clips'
clips = [f for f in os.listdir(CLIPS_PATH) if f.endswith('.mp4')]
print(f'Found {len(clips)} clips')

In [ ]:
# CELL 3 — Clone repo
!git clone https://github.com/nepitunepq/Animators.git /content/Animators
%cd /content/Animators

In [ ]:
# CELL 4 — Install dependencies
!pip install -q diffusers transformers accelerate huggingface_hub
!pip install -q bitsandbytes tqdm opencv-python

# Clone diffusion-pipe สำหรับ LoRA training
!git clone --recurse-submodules https://github.com/tdrussell/diffusion-pipe /content/diffusion-pipe
!pip install -q -r /content/diffusion-pipe/requirements.txt

In [ ]:
# CELL 5 — Copy clips จาก Drive มา local (เร็วกว่าอ่านจาก Drive ตรงๆ)
import shutil, os

# สร้างทั้งสอง folder เพื่อรองรับ config.py ทุก version
os.makedirs('/content/Animators/clips', exist_ok=True)
os.makedirs('/content/Animators/blender_clips', exist_ok=True)

for f in clips:
    src = os.path.join(CLIPS_PATH, f)
    shutil.copy2(src, f'/content/Animators/clips/{f}')
    shutil.copy2(src, f'/content/Animators/blender_clips/{f}')

print(f'Copied {len(clips)} clips to clips/ and blender_clips/')

In [ ]:
# CELL 6 — Prepare dataset (resize + captions)
%cd /content/Animators
!python dataset_prep.py

In [ ]:
# CELL 7 — Download Wan 2.1 T2V 1.3B (เล็กพอสำหรับ T4 free Colab)
# โมเดลนี้ขนาด ~5GB เท่านั้น เหมาะกับ T4 15GB VRAM
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id='Wan-AI/Wan2.1-T2V-1.3B',
    local_dir='/content/Animators/models/wan2.1',
    ignore_patterns=['*.bin'],
)
print('Model downloaded ✓')

In [ ]:
LORA_OUTPUT = '/content/drive/MyDrive/COMP_ANI/lora_output'
import os
os.makedirs(LORA_OUTPUT, exist_ok=True)

toml_content = """
output_dir = '/content/drive/MyDrive/COMP_ANI/lora_output'
save_every_n_steps = 250

[model]
type = 'wan'
ckpt_path = '/content/Animators/models/wan2.1'
dtype = 'bfloat16'

[network]
type = 'lora'
rank = 16
alpha = 16

[optimizer]
type = 'adamw8bit'
lr = 1e-4

[training]
steps = 1500
batch_size = 1
gradient_accumulation_steps = 8
gradient_checkpointing = true
seed = 42

[[datasets]]
type = 'video'
directory = '/content/Animators/wan-dataset/videos'
caption_extension = '.txt'
resolution = [480, 288]
num_frames = 17
fps = 24
"""

with open('/content/Animators/lora_config.toml', 'w') as f:
    f.write(toml_content)
print('TOML config written ✓')

In [ ]:
!pip install -q mpi4py

In [ ]:
# CELL 10 — Run training
!python /content/diffusion-pipe/train.py --config /content/Animators/lora_config.toml